# Reproducible research-data comparison with Kessetsu

This notebook runs an ordinary `.kess` circuit through the local CLI, imports an explicitly synthetic scope-style CSV, projects the typed simulation through Kessetsu Core, and compares the two datasets without reimplementing interpolation or residual metrics in Python.

> This is a software workflow example, not a physical measurement or model-validation claim.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
from kessetsu import KessetsuClient

repo = Path.cwd()
if not (repo / 'examples' / 'research' / 'rc-step.kess').exists():
    raise RuntimeError('Run this notebook from the Kessetsu repository or release-bundle root')
artifacts = repo / '.artifacts' / 'research-notebook'
artifacts.mkdir(parents=True, exist_ok=True)
client = KessetsuClient()
print(f'Kessetsu CLI {client.version()}')

In [ ]:
research = repo / 'examples' / 'research'
run = client.simulate_file(research / 'rc-step.kess')
simulation_table = run.simulation.table(0).to_pandas()
simulation_table.head()

In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

observed = client.import_csv(
    research / 'scope-style.csv',
    read_json(research / 'scope-style.kessimport.json'),
    output=artifacts / 'observed.kessdata.json',
    force=True,
)
reference = client.simulation_to_research_data(
    run,
    read_json(research / 'rc-step.kesssim.json'),
    output=artifacts / 'simulation.kessdata.json',
    force=True,
)
comparison = client.compare(
    observed,
    reference,
    read_json(research / 'rc-simulation.kesscompare.json'),
    output=artifacts / 'comparison.kesscompare.json',
    force=True,
)
comparison.metrics('out')

In [ ]:
frame = comparison.table('out').to_pandas()
units = frame.attrs['units']
fig, (signal_axis, residual_axis) = plt.subplots(2, 1, sharex=True, figsize=(8, 6))
signal_axis.plot(frame['axis'] * 1e3, frame['observed'], 'o', label='Synthetic observation')
signal_axis.plot(frame['axis'] * 1e3, frame['predicted'], '-', label='Kessetsu simulation')
signal_axis.set_ylabel(f"Voltage ({units['observed']})")
signal_axis.legend()
signal_axis.grid(alpha=0.25)
residual_axis.axhline(0, color='0.5', linewidth=1)
residual_axis.plot(frame['axis'] * 1e3, frame['residual'], 'o-')
residual_axis.set_xlabel('Time (ms)')
residual_axis.set_ylabel(f"Residual ({units['residual']})")
residual_axis.grid(alpha=0.25)
fig.suptitle('RC step: explicit synthetic data vs typed simulation')
fig.tight_layout()
fig.savefig(artifacts / 'comparison.png', dpi=160)
fig

The output directory now contains the original normalized dataset, the simulation-derived dataset, the complete point-by-point comparison and the plot. Dataset metadata distinguishes `synthetic` from `simulation`, and records the source hash, simulator identity, selected analysis and signal mapping.